# Rerank Explained Demo

This notebook explains reranking with a small, easy example.

In a RAG pipeline, retrieval usually happens in two passes:

1. **Retrieve**: quickly collect a few candidate chunks from many documents.
2. **Rerank**: score those candidates more carefully and put the best chunk first.

Think of retrieval as `find possible answers`, and reranking as `choose the best answer`.

## Why Reranking Helps

A first retriever is usually fast, but not perfect. It may return documents that share keywords with the question but do not actually answer it.

A reranker looks at each `(question, document)` pair and asks: **how useful is this document for this exact question?**

In [ ]:
from collections import Counter
import re


def tokenize(text):
    """Lowercase text and keep only word tokens."""
    return re.findall(r"[a-z]+", text.lower())


def preview(text, max_chars=120):
    text = " ".join(text.split())
    return text if len(text) <= max_chars else text[:max_chars] + "..."

## Step 1: Create A Small Corpus

We will use short chunks about France and Paris.

Some chunks mention many query words, but only one chunk directly answers the question.

In [ ]:
documents = [
    {
        "id": "doc_1",
        "text": "France is a country in Europe. It is known for wine, food, art, and history.",
    },
    {
        "id": "doc_2",
        "text": "The Eiffel Tower is one of the most famous landmarks in Paris.",
    },
    {
        "id": "doc_3",
        "text": "Paris is the capital city of France and a major center for culture and tourism.",
    },
    {
        "id": "doc_4",
        "text": "A capital can mean money used to start a business or the most important city in a region.",
    },
    {
        "id": "doc_5",
        "text": "Many visitors travel across France by train to see museums, villages, and coastlines.",
    },
]

query = "What is the capital of France?"

print("Query:", query)
print("\nDocuments:")
for doc in documents:
    print(f"{doc['id']}: {doc['text']}")

## Step 2: First Pass Retrieval

This simple retriever uses keyword overlap. It counts how many query words appear in each document.

This is intentionally simple so we can see why the first pass can be imperfect.

In [ ]:
STOPWORDS = {"what", "is", "the", "of", "a", "an", "to", "for", "and", "in"}


def retrieve_by_keyword_overlap(query, documents, top_k=4):
    query_terms = [term for term in tokenize(query) if term not in STOPWORDS]
    scored = []

    for doc in documents:
        doc_terms = tokenize(doc["text"])
        doc_counts = Counter(doc_terms)
        score = sum(doc_counts[term] for term in query_terms)
        scored.append({**doc, "retrieval_score": score})

    return sorted(scored, key=lambda item: item["retrieval_score"], reverse=True)[:top_k]


candidates = retrieve_by_keyword_overlap(query, documents, top_k=4)

print("First pass retrieval results:")
for rank, doc in enumerate(candidates, start=1):
    print(f"{rank}. {doc['id']} | retrieval_score={doc['retrieval_score']} | {preview(doc['text'])}")

## Step 3: Rerank The Candidates

Now we apply a more careful scoring rule.

For this example, the reranker rewards chunks that contain the important concepts `capital`, `france`, and `paris`. It also gives a bonus when those concepts are close together.

Real rerankers use a trained cross-encoder model or an LLM, but the idea is the same: score each candidate against the query more carefully.

In [ ]:
def simple_rerank_score(query, document_text):
    terms = tokenize(document_text)
    term_set = set(terms)

    score = 0

    # Strong signals for this exact question.
    if "capital" in term_set:
        score += 2
    if "france" in term_set:
        score += 2
    if "paris" in term_set:
        score += 4

    # Bonus when the words appear near each other.
    positions = {term: [i for i, value in enumerate(terms) if value == term] for term in ["paris", "capital", "france"]}
    if all(positions.values()):
        closest_span = min(
            max(p, c, f) - min(p, c, f)
            for p in positions["paris"]
            for c in positions["capital"]
            for f in positions["france"]
        )
        if closest_span <= 8:
            score += 3

    return score


reranked = []
for doc in candidates:
    reranked.append({**doc, "rerank_score": simple_rerank_score(query, doc["text"])})

reranked = sorted(reranked, key=lambda item: item["rerank_score"], reverse=True)

print("Reranked results:")
for rank, doc in enumerate(reranked, start=1):
    print(
        f"{rank}. {doc['id']} | retrieval_score={doc['retrieval_score']} "
        f"| rerank_score={doc['rerank_score']} | {preview(doc['text'])}"
    )

## Step 4: Compare Before And After

The retriever found possible matches. The reranker moved the best answer to the top.

This is useful because your final LLM answer is usually generated from the top few chunks. Better chunk order usually means better answers.

In [ ]:
print("Before reranking:")
for rank, doc in enumerate(candidates, start=1):
    print(f"{rank}. {doc['id']} -> {preview(doc['text'])}")

print("\nAfter reranking:")
for rank, doc in enumerate(reranked, start=1):
    print(f"{rank}. {doc['id']} -> {preview(doc['text'])}")

best_context = reranked[0]["text"]
print("\nBest context to send to the LLM:")
print(best_context)

## Optional: What A Real Cross-Encoder Reranker Looks Like

A cross-encoder takes the query and document together, then returns one relevance score.

You can uncomment and run this cell if you have `sentence-transformers` installed. The first run downloads the model.

In [ ]:
# !pip install sentence-transformers

# from sentence_transformers import CrossEncoder

# model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# pairs = [(query, doc["text"]) for doc in candidates]
# scores = model.predict(pairs)

# real_reranked = sorted(
#     [{**doc, "cross_encoder_score": float(score)} for doc, score in zip(candidates, scores)],
#     key=lambda item: item["cross_encoder_score"],
#     reverse=True,
# )

# for rank, doc in enumerate(real_reranked, start=1):
#     print(f"{rank}. {doc['id']} | score={doc['cross_encoder_score']:.4f} | {preview(doc['text'])}")

## Final Takeaway

Reranking is not a replacement for retrieval. It is a second pass after retrieval.

- Retrieval should be fast and broad.
- Reranking should be slower and more precise.
- The final answer should use the highest-ranked chunks after reranking.

Common production pattern:

```text
User question -> retrieve top 20 chunks -> rerank top 20 -> send top 3-5 chunks to the LLM
```